# Qwen3-4B — fp16 + LoRA fine-tune on Colab T4 (v9, corrected)

**Runtime this was built and verified against:** Python 3.13, torch 2.11.0+cu130,
transformers 4.57.6, peft 0.21.1, accelerate 1.15.0, Tesla T4 15.64 GB.

### Why fp16 and not QLoRA 4-bit
`bitsandbytes==0.45.5` ships a binary that does not export
`cquantize_blockwise_fp32_nf4` on Python 3.13, so every 4-bit load dies with
`AttributeError`. That is not fixable from notebook code. Qwen3-4B in fp16 is
~8.9 GB and fits the T4 with room for LoRA, activations and optimiser state.

### Why torchao is removed
`peft==0.21.1` requires `torchao > 0.16.0` and **raises** (rather than
returning `False`) when it finds the preinstalled 0.10.0, which breaks
`get_peft_model`. Uninstalling torchao makes the probe return cleanly.

### Run order
Cells 1-7 top to bottom. Cell 1 includes a zero-download preflight that catches
PEFT/accelerate breakage in seconds instead of after a 9 GB model load.

In [ ]:
import importlib, subprocess, sys

!pip -q install --no-cache-dir \
  "transformers==4.57.6" "peft==0.21.1" "datasets==4.8.5" \
  "accelerate==1.15.0" "sentencepiece" "protobuf"

# torchao 0.10.0 (preinstalled) makes peft 0.21.1 raise during get_peft_model.
# bitsandbytes 0.45.5 is broken on py3.13. Both are unused in an fp16 run.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "bitsandbytes", "torchao"], check=False)

import torch, transformers, peft, accelerate

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)

for m in ("bitsandbytes", "torchao"):
    sys.modules.pop(m, None)
    try:
        importlib.import_module(m)
        print(m, ": STILL PRESENT (unexpected)")
    except Exception:
        print(m, ": removed OK")

assert torch.cuda.is_available(), "no GPU - set Runtime > Change runtime type > T4"
props = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info()
print("GPU         :", props.name)
print("VRAM        : %.2f GB" % (total / 1e9))
print("free now    : %.2f GB" % (free / 1e9))
print("capability  : %d.%d" % (props.major, props.minor))
assert props.total_memory / 1e9 > 12, "need a T4-class GPU with >=12 GB"

### Preflight — catches PEFT breakage in seconds

This builds a 64-dim toy model **locally** (no download) and attaches a LoRA to
it. That is the exact call that raised `ImportError: incompatible torchao`, so
if the environment is wrong you find out here instead of after loading 9 GB.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import LlamaConfig, LlamaForCausalLM

toy = LlamaForCausalLM(LlamaConfig(
    hidden_size=64, intermediate_size=128, num_hidden_layers=1,
    num_attention_heads=2, num_key_value_heads=2, vocab_size=128,
))
toy = get_peft_model(toy, LoraConfig(
    r=4, lora_alpha=8, lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM",
))
n = sum(p.numel() for p in toy.parameters() if p.requires_grad)
print("peft LoRA attach: OK (%d trainable params)" % n)
del toy
torch.cuda.empty_cache()
print("PREFLIGHT PASSED - continue to the next cell")

## 2. Configuration

In [ ]:
CONFIG = {
    # ---- BASE MODEL ----
    # Option 1: clean base. Breaks the v7h/v8 stacking chain and removes any
    # inherited decline-to-act bias, giving an honest baseline.
    "base_model": "Qwen/Qwen3-4B",
    # Option 2: continue the chain from v7h.
    # "base_model": "unsloth/Qwen3-4B",

    # ---- DATA (resolved from Drive in cell 4) ----
    # Order matters: tool-calling first, refusal reduction second,
    # prompt-injection resistance third.
    "datasets": [
        "train_v8_injection.jsonl",   # 600 rows - injection vs tool-use
        "train_v6.jsonl",             # 890 rows - general assistant
        "compliance_clean.jsonl",     # 514 rows - refusal reduction
    ],
    "val_file": "val_v6.jsonl",
    "max_samples": None,

    # ---- LoRA: matches the v8 lineage (r=16, alpha=16, dropout 0.0) ----
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],

    # ---- T4-SAFE TRAINING ----
    "max_seq_length": 1024,     # lower this first if you OOM: 768
    "per_device_batch": 1,
    "grad_accum": 16,           # effective batch 16
    "learning_rate": 2e-4,
    "epochs": 1,
    # T4 (sm_75) has no FlashAttention-2 kernels; "sdpa" is the safe path.
    "attn_implementation": "sdpa",
    "max_steps": -1,            # -1 = let epochs drive it
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "lr_scheduler": "cosine",
    "logging_steps": 25,
    "save_steps": 100,
    "seed": 42,

    # ---- LOCAL + DRIVE PATHS ----
    # /content is wiped when the runtime dies, so checkpoints are mirrored to
    # Drive on every save and resume prefers Drive.
    "output_dir": "/content/qwen3-4b-v9",
    "drive_dir": "/content/drive/MyDrive/model-training/v9",
}

print("base model   :", CONFIG["base_model"])
print("datasets     :", CONFIG["datasets"])
print("val file     :", CONFIG["val_file"])
print("eff. batch   :", CONFIG["per_device_batch"] * CONFIG["grad_accum"])
print("total tokens :", CONFIG["per_device_batch"] * CONFIG["grad_accum"] * CONFIG["max_seq_length"])

## 3. Mount Drive and locate the data

Searches your whole Drive for a folder that holds every required `.jsonl`, so it
works no matter where you dropped them. You can skip this cell if you already
know the path — just set `DATA_DIR` by hand.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

NEEDED = CONFIG["datasets"] + [CONFIG["val_file"]]
ROOT = "/content/drive/MyDrive"


def find_data_dir(root, needed):
    # First directory under root holding every required .jsonl.
    if all(os.path.exists(os.path.join(root, n)) for n in needed):
        return root
    for dirpath, _dirnames, filenames in os.walk(root):
        if all(n in filenames for n in needed):
            return dirpath
    return None


DATA_DIR = find_data_dir(ROOT, NEEDED)

if DATA_DIR is None:
    raise SystemExit(
        "Could not find all of these under %s:\n\n    %s\n\n"
        "Put the files in Drive, or hardcode DATA_DIR directly below."
        % (ROOT, NEEDED)
    )

print("DATA_DIR =", DATA_DIR)
print()
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(".jsonl"):
        print("  %9.2f MB  %s" % (os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6, f))

## 4. Load the datasets

In [ ]:
import json, random
from datasets import Dataset


def load_jsonl(path):
    rows, bad = [], 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        print("  (skipped %d malformed lines in %s)" % (bad, os.path.basename(path)))
    return rows


parts, per_file = [], {}
for fn in CONFIG["datasets"]:
    rows = load_jsonl(os.path.join(DATA_DIR, fn))
    per_file[fn] = len(rows)
    parts += rows
    print("  %-32s %6d rows" % (fn, len(rows)))

if not parts:
    raise SystemExit("no training rows loaded")

random.Random(CONFIG["seed"]).shuffle(parts)
if CONFIG["max_samples"]:
    parts = parts[: CONFIG["max_samples"]]

n_tools = sum(1 for r in parts if r.get("tools"))
print("\nTOTAL TRAIN : %d rows  (%d with tool schemas)" % (len(parts), n_tools))
if n_tools == 0:
    print("WARNING: no rows carry tool schemas - tool calling cannot be learned")

train_ds = Dataset.from_list(parts)

val_ds = None
vp = os.path.join(DATA_DIR, CONFIG["val_file"])
if os.path.exists(vp):
    val_ds = Dataset.from_list(load_jsonl(vp))
    print("VAL         : %d rows" % len(val_ds))
else:
    print("VAL         : %s not found, training without eval" % CONFIG["val_file"])

print(train_ds)

## 5. Tokenizer, fp16 base model, LoRA

fp16 base is ~8.9 GB. `enable_input_require_grads()` and `use_cache=False` are
both required once gradient checkpointing is on.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation=CONFIG["attn_implementation"],
    low_cpu_mem_usage=True,
)
model.enable_input_require_grads()   # required with gradient checkpointing
model.config.use_cache = False       # required with gradient checkpointing

print("loaded:", model.config._name_or_path)
print("dtype :", next(model.parameters()).dtype)
print("VRAM  : %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

free, total = torch.cuda.mem_get_info()
print("VRAM w/ LoRA: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
print("free/total  : %.2f / %.2f GB" % (free / 1e9, total / 1e9))

## 6. Tokenize and train

Notes:

* `tools=` is passed into the chat template. The 600 tool-calling rows carry
  their schemas in a `tools` column; without this the template strips them and
  the model never learns to emit tool calls.
* Plain `Trainer` is used because this cell already tokenizes and builds
  `labels` itself, so `SFTTrainer` would only add a version surface.
* Checkpoints are mirrored to Drive **on every save**. `/content` is wiped when
  the runtime dies, and the free tier caps a session at roughly 3h20m — without
  this a long run loses everything.
* Resume picks the highest `checkpoint-N` from Drive, then from `/content`.

In [ ]:
import glob, shutil
from transformers import Trainer, TrainerCallback, TrainingArguments, DataCollatorForSeq2Seq


def _tools_for(batch, i):
    # Tool schema for row i, or None when the row has none.
    tl = batch.get("tools")
    if not isinstance(tl, list):
        return None
    return tl[i] if tl[i] else None


def tok_fn(batch):
    has_tools = "tools" in batch
    texts = [
        tok.apply_chat_template(
            m,
            tools=_tools_for(batch, i) if has_tools else None,
            tokenize=False,
            add_generation_prompt=False,
        )
        for i, m in enumerate(batch["messages"])
    ]
    # Variable length here; the collator pads each batch and sets label pads to
    # -100. Padding to max_length instead would put pad tokens into the labels
    # (measured: ~62% of all label slots) and waste most of the compute.
    enc = tok(texts, truncation=True, max_length=CONFIG["max_seq_length"],
              padding=False, add_special_tokens=False)
    # Supervise the whole turn. Qwen's template does NOT render
    # add_generation_prompt=True as a token prefix of the full conversation (it
    # appends a generation header), so masking by prompt length would cut into
    # the answer. The rendered turn already ends with the template's turn
    # terminator, which is the eos token, so stop behaviour is still learned.
    enc["labels"] = [list(ids) for ids in enc["input_ids"]]
    return enc


train_tok = train_ds.map(tok_fn, batched=True, remove_columns=train_ds.column_names)
val_tok = None
if val_ds is not None and len(val_ds):
    val_tok = val_ds.map(tok_fn, batched=True, remove_columns=val_ds.column_names)

if len(train_tok) == 0:
    raise RuntimeError("No training rows survived tokenization - "
                       "check DATA_DIR and max_seq_length.")

DEST = CONFIG["drive_dir"]
os.makedirs(DEST, exist_ok=True)


class DriveCheckpointSync(TrainerCallback):
    # Mirror each checkpoint to Drive; /content does not survive a restart.

    def __init__(self, dest, keep=3):
        self.dest, self.keep = dest, keep

    # Mirror each checkpoint to Drive; /content does not survive a restart.
    def on_save(self, args, state, control, **kwargs):
        src = os.path.join(args.output_dir, "checkpoint-%d" % state.global_step)
        if not os.path.isdir(src):
            return
        dst = os.path.join(self.dest, os.path.basename(src))
        try:
            shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
            kept = sorted(glob.glob(os.path.join(self.dest, "checkpoint-*")),
                          key=lambda p: int(p.rsplit("-", 1)[1]))
            for old in kept[:-self.keep]:
                shutil.rmtree(old, ignore_errors=True)
            print("[sync] %s -> Drive (%d kept)"
                  % (os.path.basename(src), min(len(kept), self.keep)), flush=True)
        except Exception as exc:                      # never kill the run on IO
            print("[sync] FAILED: %s" % exc, flush=True)


def latest_checkpoint(dirs):
    best = None
    for d in dirs:
        for c in glob.glob(os.path.join(d, "checkpoint-*")):
            try:
                step = int(c.rsplit("-", 1)[1])
            except ValueError:
                continue
            if best is None or step > best[0]:
                best = (step, c)
    return best[1] if best else None


RESUME = latest_checkpoint([DEST, CONFIG["output_dir"]])
print("resuming from:", RESUME) if RESUME else print("no checkpoint - starting fresh")

train_kwargs = dict(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["per_device_batch"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    num_train_epochs=CONFIG["epochs"],
    max_steps=CONFIG["max_steps"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    fp16=True,                 # GradScaler - T4 safe
    bf16=False,                # T4 has no BF16 tensor cores
    gradient_checkpointing=True,
    optim="adamw_torch",       # paged_adamw_8bit needs the broken bitsandbytes
    report_to=[],
    seed=CONFIG["seed"],
    group_by_length=True,
    remove_unused_columns=False,
)
if val_tok is not None:
    train_kwargs["eval_strategy"] = "epoch"

args = TrainingArguments(**train_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tok,
        padding=True,                # pad per batch, not to the cap
        # deliberately no max_length here: tok_fn already applies
        # truncation=True at CONFIG["max_seq_length"], so every row is <= the
        # cap. A max_length on the collator is vestigial and only emits a
        # misleading "max_length ignored, no truncation" warning.
        label_pad_token_id=-100,     # pad positions never contribute to loss
        return_tensors="pt",
    ),
    callbacks=[DriveCheckpointSync(DEST, keep=3)],
)
trainer.train(resume_from_checkpoint=RESUME)

## 7. Results and export

In [ ]:
hist = [h for h in trainer.state.log_history if "loss" in h]
if hist:
    print("initial loss:", hist[0]["loss"])
    print("final   loss:", hist[-1]["loss"])
else:
    print("no loss logged yet")

print("steps      :", trainer.state.global_step)
print("peak VRAM  : %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))

ADAPTER = os.path.join(CONFIG["output_dir"], "adapter")
os.makedirs(ADAPTER, exist_ok=True)
trainer.model.save_pretrained(ADAPTER)
tok.save_pretrained(ADAPTER)

DEST = CONFIG["drive_dir"]
os.makedirs(DEST, exist_ok=True)
dst = os.path.join(DEST, "adapter")
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(ADAPTER, dst)

print("\nadapter saved locally ->", ADAPTER)
print("adapter saved to Drive ->", dst)
print("checkpoints on Drive    ->",
      sorted(os.path.basename(c) for c in glob.glob(os.path.join(DEST, "checkpoint-*"))))

## 8. Merge and quantise (durable, Drive-backed, GPU or CPU)

Still inert by default — flip `RUN_MERGE` to `True` when you actually want it.
This cell runs on **either** a GPU or a CPU runtime.

**Run the merge on the GPU runtime**, then switch to CPU only if the quantise
step OOMs on host RAM. Merging loads the 4B base in fp16; on a CPU runtime
that is ~9 GB of host RAM and merging is slower, but it works.

Two reasons this mirrors to Drive instead of trusting `/content`:

1. `/content` is **ephemeral**. A runtime disconnect, hardware switch, or the
   one-hour session cap wipes it. Switching GPU->CPU restarts the kernel and
   destroys everything not on Drive.
2. The GGUF must be built **here**, not on your laptop. Quantising a 4B to
   Q4_K_M peaks around 11-18 GB of host RAM; your machine has ~3 GB free. A
   CPU Colab runtime gives you far more than the ~12.7 GB the T4 runtime does.

You get two durable artefacts:

- `merged/` — fp16 safetensors + tokenizer (reusable, but ~8.9 GB)
- `v9-q4km.gguf` — ~2.5 GB, the only thing your laptop actually needs to
  serve the model and match the v8 baseline pipeline

**If you already switched to CPU and lost `/content`, re-run cell 7 first** —
the merge needs `ADAPTER` and `DEST`, which come from cells 13/15.

In [ ]:
RUN_MERGE = False          # <- flip to True when you actually want the merge
COPY_MERGED_TO_DRIVE = True   # 8.9 GB safetensors; False to keep only the GGUF
RUN_QUANTIZE = True           # needs a llama.cpp build (cloned below)

if not RUN_MERGE:
    print("merge cell - inert. Set RUN_MERGE = True to enable.")
else:
    import gc, os, shutil, sys, time
    from pathlib import Path

    def _human(n):
        if n >= 1e9:
            return "%.2f GB" % (n / 1e9)
        if n >= 1e6:
            return "%.1f MB" % (n / 1e6)
        if n >= 1e3:
            return "%.1f kB" % (n / 1e3)
        return "%d B" % n

    # --- 0. recover config + adapter after a kernel restart -----------------
    # /content is ephemeral: switching GPU<->CPU restarts the kernel and wipes
    # it. Everything needed below must come back from Drive, so this cell is
    # self-sufficient and does not depend on cells 5/13/15 still being live.
    CONFIG = globals().get("CONFIG") or {
        "base_model":   "Qwen/Qwen3-4B",
        "output_dir":   "/content/qwen3-4b-v9",
        "drive_dir":    "/content/drive/MyDrive/model-training/v9",
    }
    _DRIVE = Path(CONFIG["drive_dir"])
    if not _DRIVE.exists():
        raise SystemExit(
            "Drive not mounted at %s - run cell 7 first (Runtime > restart "
            "clears /content including the mount point)." % _DRIVE)

    print("Drive contents:")
    for p in sorted(_DRIVE.glob("*")):
        n = (sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
             if p.is_dir() else p.stat().st_size)
        print("  %-24s %9s" % (p.name + ("/" if p.is_dir() else ""),
                               _human(n)))

    ADAPTER = os.path.join(CONFIG["output_dir"], "adapter")
    _drive_adapter = _DRIVE / "adapter"
    _local_adapter = Path(ADAPTER)

    if _local_adapter.exists():
        print("\nadapter present locally:", ADAPTER)
    elif _drive_adapter.exists():
        _local_adapter.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(_drive_adapter, _local_adapter)
        n = sum(f.stat().st_size for f in _local_adapter.rglob("*")
                if f.is_file())
        print("\nrecovered adapter from Drive -> %s (%.1f MB)"
              % (ADAPTER, n / 1e6))
    else:
        _ck = sorted(_DRIVE.glob("checkpoint-*"),
                     key=lambda p: int(p.name.rsplit("-", 1)[1]))
        raise SystemExit(
            "No adapter found locally or on Drive.\n"
            "  local : %s\n  drive : %s\n"
            "  checkpoints on Drive: %s\n\n"
            "If training completed, re-run cell 15 (it re-exports the adapter "
            "from the live trainer) or merge directly from the newest "
            "checkpoint instead."
            % (ADAPTER, _drive_adapter,
               [c.name for c in _ck] or "none"))

    _w = sorted(_local_adapter.glob("adapter_model.safetensors"))
    if not _w:
        raise SystemExit("adapter dir has no adapter_model.safetensors: %s"
                         % ADAPTER)
    print("weights: %s (%.1f MB)" % (_w[0].name, _w[0].stat().st_size / 1e6))

    def _free():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _vram():
        if not torch.cuda.is_available():
            return "n/a (CPU runtime)"
        return f"{torch.cuda.memory_allocated()/1e9:.2f} GB"

    def _sync(src, dst):
        # Copy file-or-dir src -> dst via a .part temp, then verify size.
        src, dst = Path(src), Path(dst)
        if not src.exists():
            print(f"  !! missing: {src}")
            return False
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.parent / (dst.name + ".part")
        t0 = time.time()
        if src.is_dir():
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(src, tmp)
            shutil.rmtree(dst, ignore_errors=True)
            tmp.rename(dst)
            n = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file())
        else:
            shutil.copy2(src, tmp)
            tmp.replace(dst)
            n = dst.stat().st_size
        ok = n > 0
        print(f"  {'OK ' if ok else 'FAIL'} {dst.name}: "
              f"{n/1e9:.2f} GB in {time.time()-t0:.0f}s")
        return ok

    # --- 1. free the training graph before loading a second copy -------------
    # Absent after a kernel restart; that is fine, there is nothing to free.
    _freed = 0
    for _n in ("trainer", "model"):
        if _n in globals():
            del globals()[_n]
            _freed += 1
    print("freed %d training object(s)" % _freed)
    _free()
    print("device after release:", _vram())

    # --- 2. merge LoRA into the base -----------------------------------------
    from transformers import AutoModelForCausalLM as _M
    from peft import PeftModel as _P

    MERGED = Path(CONFIG["output_dir"]) / "merged"
    t0 = time.time()
    # device_map={"": 0} is GPU-only; CPU runtimes must load plainly.
    _load = ({"device_map": {"": 0}} if torch.cuda.is_available() else {})
    _m = _M.from_pretrained(
        CONFIG["base_model"], dtype=torch.float16, **(_load or {})
    )
    _m = _P.from_pretrained(_m, ADAPTER).merge_and_unload()
    _m.save_pretrained(MERGED, safe_serialization=True)
    tok.save_pretrained(MERGED)
    del _m
    _free()
    print(f"merged -> {MERGED} in {time.time()-t0:.0f}s")

    # sanity: the adapter must actually have moved the weights
    if MERGED.exists():
        idx = list(MERGED.glob("*.safetensors"))
        tot = sum(f.stat().st_size for f in idx)
        print(f"  shards: {len(idx)}  total: {tot/1e9:.2f} GB")
        if tot < 1.0:
            raise RuntimeError("merge looks truncated - do not trust the output")

    # --- 3. durable copy of the merged model ---------------------------------
    if COPY_MERGED_TO_DRIVE:
        _sync(MERGED, Path(DEST) / "merged")

    # --- 4. quantise to Q4_K_M for the laptop ---------------------------------
    if RUN_QUANTIZE:
        import subprocess

        def _run(cmd):
            # subprocess rather than `!` so this survives exec()/notebooks alike.
            print("  $", " ".join(str(c) for c in cmd))
            r = subprocess.run([str(c) for c in cmd], capture_output=True,
                               text=True)
            if r.returncode != 0:
                print((r.stdout or "")[-2000:])
                print((r.stderr or "")[-2000:])
                raise RuntimeError(f"failed: {cmd[0]} (rc={r.returncode})")
            return r

        LL = Path("/content/llama.cpp")
        if not LL.exists():
            print("cloning llama.cpp (one-off, ~2 min)...")
            _run(["git", "clone", "--depth", "1", "-q",
                  "https://github.com/ggml-org/llama.cpp", str(LL)])

        BIN = LL / "build" / "bin" / "llama-quantize"
        if not BIN.exists():
            print("building llama-quantize (one-off, ~5 min)...")
            _run(["cmake", "-S", LL, "-B", LL / "build",
                  "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF",
                  "-DGGML_NATIVE=ON"])
            _run(["cmake", "--build", LL / "build", "--config", "Release",
                  "-j2", "--target", "llama-quantize"])

        F16 = Path(CONFIG["output_dir"]) / "v9-f16.gguf"
        Q4 = Path(CONFIG["output_dir"]) / "v9-q4km.gguf"
        if not F16.exists():
            t0 = time.time()
            _run([sys.executable, LL / "convert_hf_to_gguf.py", MERGED,
                  "--outfile", F16, "--outtype", "f16"])
            print(f"f16 gguf in {time.time()-t0:.0f}s "
                  f"({F16.stat().st_size/1e9:.2f} GB)")
        t0 = time.time()
        _run([BIN, F16, Q4, "Q4_K_M"])
        print(f"Q4_K_M in {time.time()-t0:.0f}s "
              f"({Q4.stat().st_size/1e9:.2f} GB)")

        _sync(Q4, Path(DEST) / "v9-q4km.gguf")
        # the f16 intermediate is ~9 GB of pure dead weight once Q4 exists
        F16.unlink(missing_ok=True)
        print("dropped f16 intermediate")
    else:
        print("quantisation skipped (RUN_QUANTIZE = False)")

    print("\nMERGE COMPLETE - re-run this cell's tail if an upload was cut off")